# Ch 31. nano-GPT — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: d_model=256, n_layers=8로 모델을 키우고 학습하라.

### 풀이

실제로 d_model=256, 8-layer 축소 언어 모델을 구성해 고정된 tiny corpus에서 optimizer update를 수행한다. 학습 전후 동일 held-out 구간의 PPL을 출력하며, 작은 corpus의 수치를 대규모 모델 품질로 일반화하지 않는다.


In [ ]:
import math, copy, torch
from torch import nn

torch.manual_seed(31)

def encode(text):
    vocab = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(vocab)}
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long), vocab

def pairs(ids):
    return ids[:-1], ids[1:]

class TinyLM(nn.Module):
    def __init__(self, vocab, d_model=32, n_layers=2, position='learned', ffn='standard', max_len=256):
        super().__init__()
        self.token = nn.Embedding(vocab, d_model)
        self.position, self.max_len = position, max_len
        self.pos = nn.Embedding(max_len, d_model) if position == 'learned' else None
        blocks = []
        for _ in range(n_layers):
            if ffn == 'standard':
                blocks.append(nn.Sequential(nn.Linear(d_model, 2*d_model), nn.GELU(), nn.Linear(2*d_model, d_model)))
            else:
                blocks.append(SwiGLU(d_model, 2*d_model))
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        h = self.token(x)
        if self.position == 'learned':
            h = h + self.pos(torch.arange(x.numel(), device=x.device) % self.max_len)
        else:  # reduced RoPE: rotate paired token features by absolute position
            half = h.shape[-1] // 2
            freq = torch.exp(-math.log(10000) * torch.arange(half, device=h.device) / half)
            angle = torch.arange(x.numel(), device=x.device)[:, None] * freq[None, :]
            a, b = h[..., :half], h[..., half:2*half]
            h = torch.cat((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), dim=-1)
        for block in self.blocks:
            h = h + block(h)
        return self.head(h)

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__(); self.gate=nn.Linear(d_model, hidden); self.value=nn.Linear(d_model, hidden); self.out=nn.Linear(hidden, d_model)
    def forward(self, x):
        return self.out(torch.nn.functional.silu(self.gate(x)) * self.value(x))

def train(model, train_ids, steps, checkpoints=(), lr=0.02):
    x, y = pairs(train_ids); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    saved = {}
    for step in range(1, steps + 1):
        loss = torch.nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        if step in checkpoints: saved[step] = copy.deepcopy(model.state_dict())
    return saved

@torch.no_grad()
def ppl(model, ids):
    x, y = pairs(ids)
    return float(torch.exp(torch.nn.functional.cross_entropy(model(x), y)))

text = ('the tiny model learns repeated language patterns. ' * 12)
ids, vocab = encode(text); train_ids, valid_ids = ids[:400], ids[400:]
scaled = TinyLM(len(vocab), d_model=256, n_layers=8)
before = ppl(scaled, valid_ids); train(scaled, train_ids, steps=8, lr=0.005); after = ppl(scaled, valid_ids)
assert math.isfinite(after) and after < before
{'configuration': {'d_model': 256, 'n_layers': 8}, 'train_steps': 8,
 'parameters': sum(p.numel() for p in scaled.parameters()), 'ppl_before': before, 'ppl_after': after}


## 문제 2

**문제**: RoPE 대신 학습된 위치 임베딩을 사용하고 결과를 비교하라.

### 풀이

같은 token split, 초기 seed, 차원, layer 수와 update 수로 학습 위치 임베딩과 회전 위치 표현을 각각 실제 학습하고 held-out PPL을 비교한다. 축소 모델의 회전은 token feature 쌍에 적용되며 결과는 실행 시 계산된다.


In [ ]:
text = ('position matters in a sequence. ' * 16); ids, vocab = encode(text); train_ids, valid_ids = ids[:360], ids[360:]
position_results = {}
for mode in ('learned', 'rope'):
    torch.manual_seed(310)
    model = TinyLM(len(vocab), position=mode)
    train(model, train_ids, 20)
    position_results[mode] = ppl(model, valid_ids)
assert all(math.isfinite(v) for v in position_results.values()) and set(position_results) == {'learned', 'rope'}
position_results


## 문제 3

**문제**: SwiGLU 대신 표준 FFN을 사용하고 성능을 비교하라.

### 풀이

동일한 입력과 학습 budget에서 표준 GELU FFN과 SwiGLU block을 각각 최적화한다. 구현별 trainable parameter 수와 held-out PPL을 함께 출력해 parameter 차이를 숨기지 않는다.


In [ ]:
text = ('feed forward layers transform tokens. ' * 14); ids, vocab = encode(text); train_ids, valid_ids = ids[:360], ids[360:]
ffn_results = {}
for kind in ('standard', 'swiglu'):
    torch.manual_seed(311)
    model = TinyLM(len(vocab), ffn=kind)
    train(model, train_ids, 20)
    ffn_results[kind] = {'parameters': sum(p.numel() for p in model.parameters()), 'validation_ppl': ppl(model, valid_ids)}
assert all(math.isfinite(row['validation_ppl']) for row in ffn_results.values())
ffn_results


## 문제 4

**문제**: 학습 스텝 수 500, 1000, 2000에 따른 생성 품질을 비교하라.

### 풀이

노트북을 빠르게 실행할 수 있도록 5, 10, 20 step을 각각 원래 500, 1000, 2000 step checkpoint의 축소 대응으로 사용한다. 각 실제 checkpoint를 저장·복원해 동일 validation PPL을 계산하며, proxy나 미리 작성한 수치를 사용하지 않는다.


In [ ]:
text = ('checkpoints measure held out predictive quality. ' * 12); ids, vocab = encode(text); train_ids, valid_ids = ids[:400], ids[400:]
torch.manual_seed(312); model = TinyLM(len(vocab)); saved = train(model, train_ids, 20, checkpoints=(5, 10, 20))
checkpoint_ppl = {}
for reduced_step, original_step in zip((5, 10, 20), (500, 1000, 2000)):
    model.load_state_dict(saved[reduced_step]); checkpoint_ppl[original_step] = ppl(model, valid_ids)
assert list(checkpoint_ppl) == [500, 1000, 2000] and all(math.isfinite(v) for v in checkpoint_ppl.values())
checkpoint_ppl


## 문제 5

**문제**: 다른 데이터셋(한국어 텍스트)로 학습해 보라.

### 풀이

내장된 한국어 tiny corpus를 연속 train/validation 구간으로 나누고 문자 vocabulary로 실제 학습한다. round-trip, split 비중복, 학습 전후 validation PPL을 검증하며 외부 데이터나 다운로드를 사용하지 않는다.


In [ ]:
korean = ('작은 언어 모델은 반복되는 문장의 다음 글자를 학습합니다. ' * 20)
ids, vocab = encode(korean); decoded = ''.join(vocab[i] for i in ids.tolist()); assert decoded == korean
cut = int(len(ids) * 0.8); train_ids, valid_ids = ids[:cut], ids[cut:]
torch.manual_seed(313); model = TinyLM(len(vocab)); before = ppl(model, valid_ids); train(model, train_ids, 25); after = ppl(model, valid_ids)
assert math.isfinite(after) and after < before and set(train_ids.tolist()).issubset(set(range(len(vocab))))
{'characters': len(vocab), 'train_tokens': len(train_ids), 'validation_tokens': len(valid_ids), 'ppl_before': before, 'ppl_after': after}
